# Dynamically build the agent team based on the specific AM alloy research task on the fly

## Notes:
> - Access to Thermo-Calc, TC-python and OpenAI API key are needed.


#### Bo Ni, March 26, 2026

## 0. Check the floor

In [1]:
# # make into one block
# # 
# {
#     "tags": [
#         "hide-input",
#     ]
# }
# # 
#@CHech the base
import os, sys
print('Here is : \n', os.popen('pwd').read())
print('What we get in hardware: \n', os.popen('nvidia-smi').read())
kernel_name = os.path.basename(sys.executable.replace("/bin/python",""))
print ("The VirEnv kernal in action: ", kernel_name)
# 
# # make it work for both py and notebook
# try: 
#     from jupyter_client import kernelspec
#     spec = kernelspec.get_kernel_spec(kernel_name)
#     print("Path to it: ", spec.resource_dir)
# except:
#     print ("This suppose to be a .py run")
# # /path/to/my/kernel
try: 
    import torch
    print("What we have in software: \n Torch version:", torch.__version__) 
    print('Python: ', sys.version) # no switch case code
    # 
    print('What hardware the software see:')
    device = torch.device(
        "cuda:0" if torch.cuda.is_available() else "cpu"
    )
    print(device)
    num_of_gpus = torch.cuda.device_count()
    print("# of GPU", num_of_gpus)
    # 
    torch.cuda.empty_cache()
except:
    print ("No torch founded")
# 

Here is : 
 /trace/group/tmousavi/bni2/1_JupyterGit/workspace/TC_Python_LLM_1/7_test_GroupAgent

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
What we get in hardware: 
 Thu Mar 26 22:25:57 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A40                     On  | 00000000:25:00.0

In [2]:
## for TC-python
# updated to 2024a

os.environ["TC24A_HOME"] = "/trace/group/tmousavi/software/Thermo-Calc/2024a"
os.environ["LSHOST"]="172.24.116.37" # "128.2.112.37"

## Working part

In [3]:
# load in basis

import autogen

import shutil
import os

## 1. Prepare the basics

In [4]:
CKeys={}

CKeys['Debug_Level'] = 1 # 0: executing mode

In [5]:
# add folders for debug purpose

wk_dir = "../Local_Store/For_7_4_0/"

print (
    os.path.exists(wk_dir)
)

False


In [6]:
# clean everything

if os.path.exists(wk_dir):
    shutil.rmtree(wk_dir)
    print (f"Remove everything...")

In [7]:
# back and create a clean path

if not os.path.exists(wk_dir):
    print (f"Create working path...")
    os.mkdir(wk_dir)

Create working path...


In [8]:
# path arrangement
# 
code_dir = wk_dir + "coding/"
vedb_dir = wk_dir + "vector_db/"

### 1.2 LLM configurations

In [9]:
config_file= "../Local_Store/Simple_OpenAI_Keys_Safe.json"


# for Agents

# config_list = autogen.config_list_from_json("OAI_CONFIG_LIST")

# ref:
# https://github.com/microsoft/autogen/blob/main/autogen/oai/openai_utils.py
config_list_agen = autogen.config_list_from_json(
    env_or_file=config_file, # "Simple_OpenAI_Keys_Safe.json",
    # file_location="../Local_Store/",
    filter_dict={
        "model": [
            "gpt-4o-2024-05-13",
            # "gpt-4o-2024-08-06",
            # "gpt-4o-mini",
        ],
    },
)

assert len(config_list_agen) > 0
print(
    "models to use: ", 
    [config_list_agen[i]["model"] for i in range(len(config_list_agen))]
)

llm_config_agen = {
    "config_list": config_list_agen, 
    "timeout": 600, 
    "temperature": 0.8, 
    "seed": 1234
}

models to use:  ['gpt-4o-2024-05-13']


In [10]:
# # config. for tools

llm_config_tool = autogen.LLMConfig.from_json(
    path= config_file, # "OAI_CONFIG_LIST", 
    timeout=120
).where(
    tags=["search"]  # comment out where to get all
)

print (llm_config_tool)

LLMConfig(timeout=120, config_list=[{'api_type': 'openai', 'model': 'gpt-4.1-2025-04-14', 'api_key': '**********', 'tags': ['search']}])


### 1.3 coding environment

In [11]:
# coding configu

from types import SimpleNamespace

# Create a SimpleNamespace object with attributes
venv_SimNam = SimpleNamespace(
    env_dir="/trace/group/tmousavi/bni2/VirEnv/12_LLM_Modeling", 
    env_name="/trace/group/tmousavi/bni2/VirEnv/12_LLM_Modeling",
    bin_path="/trace/group/tmousavi/bni2/VirEnv/12_LLM_Modeling/bin/"
)

In [12]:
# the surface one that run the codes and functions

code_exe_config = {
    # the executor to run the generated code
    # ++
    # "executor": jupyter.JupyterCodeExecutor()
    # -- it's shell, no environment is activated
    "executor": autogen.coding.LocalCommandLineCodeExecutor(
        # virtual_env_context="/trace/home/bni2/Group_WSpace_250GB/group_folder/bni2/VirEnv/6_LLM_Modelling",
        # ++
        virtual_env_context= venv_SimNam,
        work_dir= code_dir, # wk_dir + "coding"
    ),
}  # set to False if you don't want to execute the code

## 2. Team up the agents

In [13]:
# prepare for debugging

from importlib import reload

In [14]:
#  customized agents

import GroupAgent_dev.groupagent as GA

reload(GA)

__name__: GroupAgent_dev.agent_builder


<module 'GroupAgent_dev.groupagent' from '/trace/group/tmousavi/bni2/1_JupyterGit/workspace/TC_Python_LLM_1/7_test_GroupAgent/GroupAgent_dev/groupagent.py'>

In [15]:
# only for agents


adjust_nested_config_for_agents = {
    "autobuild_init_config": {
        "config_file_or_env": config_file,
        "builder_model": "gpt-4o-2024-08-06", # "gpt-4o",
        "agent_model": "gpt-4o-2024-08-06", # "gpt-4o",
    },
    "autobuild_build_config": {
        "default_llm_config": {
            "temperature": 1, 
            "top_p": 0.95, 
            "max_tokens": 2048
        },
        "code_execution_config": {
            "timeout": 300,
            "work_dir": "0_0_groupchat", # may change this for deployment
            "last_n_messages": 1,
            "use_docker": False,
        },
        "coding": True,
    },
    "group_chat_config": {"max_round": 30}, # 10
    "group_chat_llm_config": None,
    "max_turns": 10, # 5
    # "autobuild_tool_config": {
    #     'tool_root': 'default',
    # }
}

In [16]:
# on AUTOBUILD_SYSTEM_MESSAGE
# for assistant

sys_message_for_autobuild_1_team_rag_v0 ="""# Your role
You are a perfect manager of a group of advanced experts.

# How to solve the task
When a task is assigned to you, first explicitly STATE how to do the following:
1. Analysis of its constraints and conditions for completion.
2. Respond with a specific plan of how to solve the task.

After that, you can solve the task in two ways:
- Delegate the resolution of tasks to other experts created by seeking a group of experts for help and derive conclusive insights from their conversation summarization.
- Analysis and solve the task with your coding and language skills.

# How to seek experts help
The tool "seek_experts_help" can build a group of experts according to the building_task and let them chat with each other in a group chat to solve the execution_task you provided.
- This tool will summarize the essence of the experts' conversation and the derived conclusions.
- You should not modify any task information from meta_user_proxy, including code blocks, but you can provide extra information.
- Within a single response, you are limited to initiating one group of experts.

## building_task
This task helps a build manager to build a group of experts for your task.
You should suggest more than one and less then four roles (including a checker for verification) with the following format.
When provided, you prefer the agents from the provided library.

### Format
- [Detailed description for role 1]
- [Detailed description for role 2]
- [Detailed description for checker]

## execution_task
This is the task that needs the experts to solve by conversation.
You should Provide the following information in markdown format.

### Format
## Task description
...
## Plan for solving the task
...
## Output format
...
## Constraints and conditions for completion
...
## [Optional] results (including code blocks) and reason from last response
...

# After seek_experts_help
You will receive a comprehensive conclusion from the conversation, including the task information, results, reason for the results, conversation contradiction or issues, and additional information.
You **must** conduct a thorough verification for the result and reason's logical compliance by leveraging the step-by-step backward reasoning with the same group of experts (with the same group name) when:
- The conversation has contradictions or issues (need double-check marked as yes), or
- The result is different from the previous results.

Note that the previous experts will forget everything after you obtain the response from them. You should provide the results (including code blocks) you collected from the previous experts' response and put it in the new execution_task.

# Some useful instructions
- You only have one tool called "seek_experts_help".
- Provide a answer yourself after "seek_experts_help".
- You should suggest python code in a python coding block (```python...```). If you need to get the value of a variable, you must use the print statement.
- When using code, you must indicate the script type in the code block.
- Do not suggest incomplete code which requires users to modify.
- Be clear about which step uses code, which step uses your language skill, and which step to build a group chat.
- If the code's result indicates there is an error, fix the error and output the whole code again.
- If the error can't be fixed or if the task is not solved even after the code is executed successfully, analyze the problem, revisit your assumption, collect additional info you need, and think of a different approach to try.
- Include verifiable evidence in your response if possible.
- After completing all tasks and verifications, you should conclude the operation and reply "TERMINATE"
"""

In [17]:
# on AUTOBUILD_SYSTEM_MESSAGE
# for assistant
# 
# to use web search related
# modified and based on http://127.0.0.1:8893/lab/tree/workspace/TC_Python_LLM_1/6_test_group_agents/1_add_pins_in_code_3.ipynb


sys_message_for_autobuild_v1 ="""# Your role
You are a perfect manager of a group of advanced experts.

# How to solve the task
When a task is assigned to you, FIRST explicitly state how to do the following:
1. Analysis of its constraints and conditions for completion.
2. Respond with a specific plan of how to solve the task.

After that, you can solve the task in two ways:
- Delegate the resolution of tasks to other experts created by seeking a group of experts for help and derive conclusive insights from their conversation summarization.
- Analysis and solve the task with your coding and language skills.

# How to seek experts help
The tool "seek_experts_help" can build a group of experts according to the building_task and let them chat with each other in a group chat to solve the execution_task you provided.
- This tool will summarize the essence of the experts' conversation and the derived conclusions.
- You should not modify any task information from meta_user_proxy, including code blocks, but you can provide extra information.
- Within a single response, you are limited to initiating one group of experts.

## building_task
This task helps a build manager to build a group of experts for your task.
You should suggest more than one roles (including a web searcher for getting up-to-date research and debug info and a checker for verification) with the following format.
You don't suggest too many roles that waste the resource.

### Format
- [Detailed description for web searcher]
- [Detailed description for role 1]
- [Detailed description for role 2]
- [Detailed description for checker]

## execution_task
This is the task that needs the experts to solve by conversation.
You should Provide the following information in markdown format.

### Format
## Task description
...
## Plan for solving the task
...
## Output format
...
## Constraints and conditions for completion
...
## [Optional] results (including code blocks) and reason from last response
...

# After seek_experts_help
You will receive a comprehensive conclusion from the conversation, including the task information, results, reason for the results, conversation contradiction or issues, and additional information.
You **must** conduct a thorough verification for the result and reason's logical compliance by leveraging the step-by-step backward reasoning with the same group of experts (with the same group name) when:
- The conversation has contradictions or issues (need double-check marked as yes), or
- The result is different from the previous results.

Note that the previous experts will forget everything after you obtain the response from them. You should provide the results (including code blocks) you collected from the previous experts' response and put it in the new execution_task.

# Some useful instructions
- You only have one tool called "seek_experts_help".
- Provide a answer yourself after "seek_experts_help".
- You should suggest python code in a python coding block (```python...```). If you need to get the value of a variable, you must use the print statement.
- When using code, you must indicate the script type in the code block.
- Do not suggest incomplete code which requires users to modify.
- Be clear about which step uses code, which step uses your language skill, and which step to build a group chat.
- If the code's result indicates there is an error, fix the error and output the whole code again.
- If the error can't be fixed or if the task is not solved even after the code is executed successfully, analyze the problem, revisit your assumption, collect additional info you need, and think of a different approach to try.
- Include verifiable evidence in your response if possible.
- After completing all tasks and verifications, you should conclude the operation and reply "TERMINATE"
"""

In [18]:
# user proxy

group_user_proxy = autogen.UserProxyAgent(
    name="group_user_proxy", 
    human_input_mode="NEVER",
    # ++
    code_execution_config=code_exe_config,
    max_consecutive_auto_reply=5,
)

In [19]:
# build the agent team


group_agent_1 = GA.GroupAgent(
    name="group_agent_2",
    llm_config=llm_config_agen,
    # 
    code_execution_config=code_exe_config,
    # agent_lib="./groupagent_expert_lib.json",
    agent_config_save_path=wk_dir,  # If you'd like to save the created agents in nested chat for further use, specify the save directory here
    # ++
    # adjust to local envi
    nested_config= adjust_nested_config_for_agents,
    system_message=sys_message_for_autobuild_v1,
    # on agents
    agent_lib="../6_test_group_agents/GA_alloy_design_team_4_1_Only_WebSearch.json"
    # "./test_json/groupagent_alloy_design_team_2_RAG.json", # 1: no tools
    # on tools
    # tool_lib="/trace/group/tmousavi/bni2/1_JupyterGit/workspace/TC_Python_LLM_1/6_test_group_agents/GroupAgent_dev/user_tools",
)

Debug level: 1


## 3. Test

### 3.1 on the knowledge retrivial

In [20]:
# search and summarize

test_statement_1 = """\
Search the web for recent papers on refractory alloy design for additive manufacturing and summarize the new research trends.
"""

In [32]:
# search, summarize and propose new ideas.


test_statement_2 = """\
Search the web for research papers in the most recent (up to 2026) five years on refractory alloy design for additive manufacturing, 
summarize the research emerging trends, understand them, 
then analyze their remaining gaps, identify the correspondingly overlooked opportunities, 
then propose novel and promising research directions to address these.
"""

In [33]:
result_2 = group_user_proxy.initiate_chat(
    group_agent_1, 
    message= test_statement_2,
)

group_user_proxy (to group_agent_2):

Search the web for research papers in the most recent (up to 2026) five years on refractory alloy design for additive manufacturing, 
summarize the research emerging trends, understand them, 
then analyze their remaining gaps, identify the correspondingly overlooked opportunities, 
then propose novel and promising research directions to address these.


--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
Team_builder (to GroupAgent):

Search the web for research papers in the most recent (up to 2026) five years on refractory alloy design for additive manufacturing, 
summarize the research emerging trends, understand them, 
then analyze their remaining gaps, identify the correspondingly overlooked opportunities, 
then propose novel and 

==> Generating agents...
['Materials_Science_Expert'] are generated.
==> Generating system message...
Preparing system message for Materials_Science_Expert
==> Generating description...
Preparing description for Materials_Science_Expert
==> Creating agents...
Creating agent Materials_Science_Expert...
==> Generating agents...
['Additive_Manufacturing_Expert'] are generated.
==> Generating system message...
Preparing system message for Additive_Manufacturing_Expert
==> Generating description...
Preparing description for Additive_Manufacturing_Expert
==> Creating agents...
Creating agent Additive_Manufacturing_Expert...
==> Generating agents...
['Research_Expert'] are generated.
==> Generating system message...
Preparing system message for Research_Expert
==> Generating description...
Preparing description for Research_Expert
==> Creating agents...
Creating agent Research_Expert...
['Web_Search_Expert', 'Materials_Science_Expert', 'Additive_Manufacturing_Expert', 'Research_Expert'] are s

Creating agent Materials_Science_Expert...
Creating agent Additive_Manufacturing_Expert...
Creating agent Research_Expert...
'coding' is required, so create an 'Computer_terminal using UserProxyAgent' 
Adding user console proxy...
find tools:
tool name: web_search_preview
add 'WebSearchPreviewTool' to the userproxy
Web_Search_Expert (to chat_manager):

You are given: (1) a task and advises from your manager with a specific plan and (2) a general task.
Collect information from the general task, follow the suggestions from manager to solve the task.

# General Task
Search the web for research papers in the most recent (up to 2026) five years on refractory alloy design for additive manufacturing, 
summarize the research emerging trends, understand them, 
then analyze their remaining gaps, identify the correspondingly overlooked opportunities, 
then propose novel and promising research directions to address these.


# Task and suggestions from manager
## Task description
Search the web for